## Experiment 9: Generate Summary using RNN

In [18]:
# 🔹 Step 1: Import Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [19]:
# 🔹 Step 2: Sample Dataset
texts = [
    "deep learning is a subset of machine learning",
    "rnn is useful for sequence modeling tasks",
    "cnn is mainly used for image processing"
]

summaries = [
    "deep learning model",
    "rnn for sequence",
    "cnn for images"
]

In [20]:
# 🔹 Step 3: Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts + summaries)

input_seq = tokenizer.texts_to_sequences(texts)
target_seq = tokenizer.texts_to_sequences(summaries)

In [21]:
# 🔹 Step 4: Padding
max_len_input = max(len(x) for x in input_seq)
max_len_target = max(len(x) for x in target_seq)

input_seq = pad_sequences(input_seq, maxlen=max_len_input, padding='post')
target_seq = pad_sequences(target_seq, maxlen=max_len_target, padding='post')

In [22]:
# 🔹 Step 5: Prepare Decoder Input & Output (IMPORTANT FIX)
decoder_input = target_seq[:, :-1]
decoder_output = target_seq[:, 1:]

decoder_output = np.expand_dims(decoder_output, -1)

In [23]:
# 🔹 Step 6: Model Parameters
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 64
latent_dim = 64

In [24]:
# 🔹 Step 7: Encoder
encoder_inputs = Input(shape=(max_len_input,))
enc_emb = Embedding(vocab_size, embedding_dim)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

In [25]:
# 🔹 Step 8: Decoder (FIXED SHAPE)
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [26]:
# 🔹 Step 9: Build Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [27]:
# 🔹 Step 10: Compile Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 8, 64)     │      1,344 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, None, 64)  │      1,344 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_6[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, None,     │     33,024 │ embedding_7[0][0… │
│                     │ 64), (None, 64),  │            │ lstm_6[0][1],     │
│                     │ (None, 64)]       │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 21)  │      1,365 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 70,101 (273.83 KB)

 Trainable params: 70,101 (273.83 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# 🔹 Step 11: Train Model
model.fit([input_seq, decoder_input],
          decoder_output,
          epochs=200,
          batch_size=2)

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.0000e+00 - loss: 3.0441
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3333 - loss: 3.0231
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3333 - loss: 3.0000
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3333 - loss: 2.9772
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3333 - loss: 2.9506
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3333 - loss: 2.9193
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3333 - loss: 2.8813
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3333 - loss: 2.8339
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3333 - loss: 2.7735
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3333 - loss: 2.7088
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3333 - loss: 2.5970
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3333 

In [29]:
# 🔹 Step 12: Testing (Generate Summary)

reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}

def generate_summary(input_text):
    seq = tokenizer.texts_to_sequences([input_text])
    seq = pad_sequences(seq, maxlen=max_len_input, padding='post')

    # start with first word (simple approach)
    target_seq_test = np.zeros((1, max_len_target-1))

    prediction = model.predict([seq, target_seq_test])
    predicted_indices = np.argmax(prediction[0], axis=1)

    summary = []
    for idx in predicted_indices:
        word = reverse_word_index.get(idx, '')
        summary.append(word)

    return " ".join(summary)

In [30]:
# 🔹 Test Example
print(generate_summary("deep learning is powerful"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
for sequence
